In [1]:
import pandas as pd 
import numpy as np

df_all_games = pd.read_csv('data/games.csv')
df_one_row_per_game = pd.read_csv('data/one-row-per-game.csv')

In [2]:
df_all_games.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22024,1610612738,BOS,Boston Celtics,22401187,2025-04-13,BOS vs. CHA,W,241,93,...,0.692,15,34,49,22,12,6,13,14,7.0
1,22024,1610612747,LAL,Los Angeles Lakers,22401199,2025-04-13,LAL @ POR,L,239,81,...,0.714,13,29,42,21,9,11,20,21,-28.0
2,22024,1610612759,SAS,San Antonio Spurs,22401197,2025-04-13,SAS vs. TOR,W,240,125,...,0.875,9,41,50,22,11,2,12,13,7.0
3,22024,1610612754,IND,Indiana Pacers,22401189,2025-04-13,IND @ CLE,W,289,126,...,0.909,14,47,61,27,15,10,13,20,8.0
4,22024,1610612760,OKC,Oklahoma City Thunder,22401196,2025-04-13,OKC @ NOP,W,241,115,...,0.889,17,37,54,16,8,4,10,25,15.0


In [3]:
df_one_row_per_game.head()

,GAME_ID,GAME_DATE,HOME_TEAM,PTS_home,AWAY_TEAM,PTS_away,IS_NEUTRAL,HOME_WIN,HOME_MARGIN
0,22401187,2025-04-13,BOS,93,CHA,86,False,1,7
1,22401197,2025-04-13,SAS,125,TOR,118,False,1,7
2,22401196,2025-04-13,NOP,100,OKC,115,False,0,-15
3,22401190,2025-04-13,MIA,118,WAS,119,False,0,-1
4,22401189,2025-04-13,CLE,118,IND,126,False,0,-8


In [4]:
win_pct_df = df_all_games.groupby('TEAM_ABBREVIATION')['WL'].apply(lambda x: (x=='W').mean())
win_pct_df

TEAM_ABBREVIATION
ATL    0.487805
BKN    0.317073
BOS    0.743902
CHA    0.231707
CHI    0.475610
CLE    0.780488
DAL    0.475610
DEN    0.609756
DET    0.536585
GSW    0.585366
HOU    0.634146
IND    0.609756
LAC    0.609756
LAL    0.609756
MEM    0.585366
MIA    0.451220
MIL    0.585366
MIN    0.597561
NOP    0.256098
NYK    0.621951
OKC    0.829268
ORL    0.500000
PHI    0.292683
PHX    0.439024
POR    0.439024
SAC    0.487805
SAS    0.414634
TOR    0.365854
UTA    0.207317
WAS    0.219512
Name: WL, dtype: float64

In [5]:
df_one_row_per_game["HOME_TEAM_FULL_SEASON_win_pct"] = df_one_row_per_game['HOME_TEAM'].map(win_pct_df)
df_one_row_per_game["AWAY_TEAM_FULL_SEASON_win_pct"] = df_one_row_per_game['AWAY_TEAM'].map(win_pct_df)

In [6]:
df_one_row_per_game.head()

,GAME_ID,GAME_DATE,HOME_TEAM,PTS_home,AWAY_TEAM,PTS_away,IS_NEUTRAL,HOME_WIN,HOME_MARGIN,HOME_TEAM_FULL_SEASON_win_pct,AWAY_TEAM_FULL_SEASON_win_pct
0,22401187,2025-04-13,BOS,93,CHA,86,False,1,7,0.743902,0.231707
1,22401197,2025-04-13,SAS,125,TOR,118,False,1,7,0.414634,0.365854
2,22401196,2025-04-13,NOP,100,OKC,115,False,0,-15,0.256098,0.829268
3,22401190,2025-04-13,MIA,118,WAS,119,False,0,-1,0.451220,0.219512
4,22401189,2025-04-13,CLE,118,IND,126,False,0,-8,0.780488,0.609756


In [7]:
df_one_row_per_game["FULL_SEASON_pred"] = df_one_row_per_game["HOME_TEAM_FULL_SEASON_win_pct"] >= df_one_row_per_game["AWAY_TEAM_FULL_SEASON_win_pct"]
df_one_row_per_game.head()

,GAME_ID,GAME_DATE,HOME_TEAM,PTS_home,AWAY_TEAM,PTS_away,IS_NEUTRAL,HOME_WIN,HOME_MARGIN,HOME_TEAM_FULL_SEASON_win_pct,AWAY_TEAM_FULL_SEASON_win_pct,FULL_SEASON_pred
0,22401187,2025-04-13,BOS,93,CHA,86,False,1,7,0.743902,0.231707,True
1,22401197,2025-04-13,SAS,125,TOR,118,False,1,7,0.414634,0.365854,True
2,22401196,2025-04-13,NOP,100,OKC,115,False,0,-15,0.256098,0.829268,False
3,22401190,2025-04-13,MIA,118,WAS,119,False,0,-1,0.451220,0.219512,True
4,22401189,2025-04-13,CLE,118,IND,126,False,0,-8,0.780488,0.609756,True


In [8]:
(df_one_row_per_game["FULL_SEASON_pred"] == df_one_row_per_game["HOME_WIN"]).mean()

np.float64(0.7)

##  Calculating rolling win %

In [9]:
df_all_games["GAME_DATE"]

0       2025-04-13
1       2025-04-13
2       2025-04-13
3       2025-04-13
4       2025-04-13
           ...    
2455    2024-10-23
2456    2024-10-22
2457    2024-10-22
2458    2024-10-22
2459    2024-10-22
Name: GAME_DATE, Length: 2460, dtype: str

In [10]:
# sort oldest to newest
df_all_games['GAME_DATE'] = pd.to_datetime(df_all_games['GAME_DATE'])
df_all_games = df_all_games.sort_values(by="GAME_DATE")

In [11]:
df_all_games['WIN_OR_LOSS'] = (df_all_games['WL']== 'W').astype(int)
df_all_games[['WIN_OR_LOSS', 'WL']]


,WIN_OR_LOSS,WL
2459,1,W
2456,0,L
2458,0,L
2457,1,W
2455,0,L
...,...,...
27,1,W
28,1,W
29,0,L
15,1,W


In [12]:
# create games_before cumulative sum

df_all_games['GAMES_BEFORE'] = df_all_games.groupby('TEAM_ABBREVIATION').cumcount()
df_all_games[df_all_games['TEAM_ABBREVIATION'] == 'BOS']['GAMES_BEFORE']

2459     0
2433     1
2398     2
2369     3
2345     4
        ..
130     77
102     78
85      79
43      80
0       81
Name: GAMES_BEFORE, Length: 82, dtype: int64

In [13]:
df_all_games['WINS_BEFORE'] = df_all_games.groupby('TEAM_ABBREVIATION')['WIN_OR_LOSS'].cumsum()
df_all_games['WINS_BEFORE'] = df_all_games['WINS_BEFORE'] - df_all_games['WIN_OR_LOSS']
df_all_games[df_all_games['TEAM_ABBREVIATION'] == 'BOS'].head(30)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,WIN_OR_LOSS,GAMES_BEFORE,WINS_BEFORE
2459,22024,1610612738,BOS,Boston Celtics,22400061,2024-10-22,BOS vs. NYK,W,240,132,...,40,33,6,3,3,15,23.0,1,0,0
2433,22024,1610612738,BOS,Boston Celtics,22400073,2024-10-24,BOS @ WAS,W,240,122,...,49,21,7,3,14,19,20.0,1,1,1
2398,22024,1610612738,BOS,Boston Celtics,22400089,2024-10-26,BOS @ DET,W,239,124,...,37,19,10,6,10,16,6.0,1,2,2
2369,22024,1610612738,BOS,Boston Celtics,22400104,2024-10-28,BOS vs. MIL,W,242,119,...,38,24,5,5,12,18,11.0,1,3,3
2345,22024,1610612738,BOS,Boston Celtics,22400119,2024-10-30,BOS @ IND,L,266,132,...,51,23,11,5,14,19,-3.0,0,4,4
2309,22024,1610612738,BOS,Boston Celtics,22400132,2024-11-01,BOS @ CHA,W,242,124,...,41,21,12,5,10,13,15.0,1,5,4
2285,22024,1610612738,BOS,Boston Celtics,22400141,2024-11-02,BOS @ CHA,W,240,113,...,44,23,4,4,15,17,10.0,1,6,5
2273,22024,1610612738,BOS,Boston Celtics,22400157,2024-11-04,BOS @ ATL,W,239,123,...,50,27,15,7,12,14,30.0,1,7,6
2238,22024,1610612738,BOS,Boston Celtics,22400172,2024-11-06,BOS vs. GSW,L,240,112,...,47,22,5,3,12,18,-6.0,0,8,7
2209,22024,1610612738,BOS,Boston Celtics,22400187,2024-11-08,BOS vs. BKN,W,266,108,...,39,23,7,9,13,15,4.0,1,9,7


In [14]:
df_all_games['WIN_PCT_BEFORE'] = df_all_games['WINS_BEFORE'] / df_all_games['GAMES_BEFORE']
df_all_games[df_all_games['TEAM_ABBREVIATION'] == 'BOS'].tail(3)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,AST,STL,BLK,TOV,PF,PLUS_MINUS,WIN_OR_LOSS,GAMES_BEFORE,WINS_BEFORE,WIN_PCT_BEFORE
85,22024,1610612738,BOS,Boston Celtics,22401156,2025-04-09,BOS @ ORL,L,238,76,...,20,5,5,15,11,-20.0,0,79,59,0.746835
43,22024,1610612738,BOS,Boston Celtics,22401174,2025-04-11,BOS vs. CHA,W,240,130,...,35,9,8,10,11,36.0,1,80,59,0.737500
0,22024,1610612738,BOS,Boston Celtics,22401187,2025-04-13,BOS vs. CHA,W,241,93,...,22,12,6,13,14,7.0,1,81,60,0.740741


In [15]:
df_all_games[df_all_games['GAME_ID']==22401187]

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,AST,STL,BLK,TOV,PF,PLUS_MINUS,WIN_OR_LOSS,GAMES_BEFORE,WINS_BEFORE,WIN_PCT_BEFORE
22,22024,1610612766,CHA,Charlotte Hornets,22401187,2025-04-13,CHA @ BOS,L,240,86,...,23,10,4,17,16,-7.0,0,81,19,0.234568
0,22024,1610612738,BOS,Boston Celtics,22401187,2025-04-13,BOS vs. CHA,W,241,93,...,22,12,6,13,14,7.0,1,81,60,0.740741


In [16]:
lookup = df_all_games[['GAME_ID', 'TEAM_ABBREVIATION', 'WIN_PCT_BEFORE']]

home_lookup = lookup.rename(columns={
    'TEAM_ABBREVIATION': 'HOME_TEAM',
    'WIN_PCT_BEFORE': 'WIN_PCT_BEFORE_HOME',
})

df_one_row_per_game = df_one_row_per_game.merge(
    home_lookup, on=['GAME_ID', 'HOME_TEAM'], how='left'
)

away_lookup = lookup.rename(columns={
    'TEAM_ABBREVIATION': 'AWAY_TEAM',
    'WIN_PCT_BEFORE': 'WIN_PCT_BEFORE_AWAY',
})

df_one_row_per_game = df_one_row_per_game.merge(
    away_lookup, on=['GAME_ID', 'AWAY_TEAM'], how='left'
)

In [17]:
df_one_row_per_game.shape

(1230, 14)

In [18]:
df_one_row_per_game.head()

,GAME_ID,GAME_DATE,HOME_TEAM,PTS_home,AWAY_TEAM,PTS_away,IS_NEUTRAL,HOME_WIN,HOME_MARGIN,HOME_TEAM_FULL_SEASON_win_pct,AWAY_TEAM_FULL_SEASON_win_pct,FULL_SEASON_pred,WIN_PCT_BEFORE_HOME,WIN_PCT_BEFORE_AWAY
0,22401187,2025-04-13,BOS,93,CHA,86,False,1,7,0.743902,0.231707,True,0.740741,0.234568
1,22401197,2025-04-13,SAS,125,TOR,118,False,1,7,0.414634,0.365854,True,0.407407,0.370370
2,22401196,2025-04-13,NOP,100,OKC,115,False,0,-15,0.256098,0.829268,False,0.259259,0.827160
3,22401190,2025-04-13,MIA,118,WAS,119,False,0,-1,0.451220,0.219512,True,0.456790,0.209877
4,22401189,2025-04-13,CLE,118,IND,126,False,0,-8,0.780488,0.609756,True,0.790123,0.604938


In [19]:
# if eithe team has no history, predict home team; otherwise, predict based on win %, tie-break with home team
df_one_row_per_game['rolling_history_pred'] = (
    np.where(
        df_one_row_per_game['WIN_PCT_BEFORE_HOME'].isna() | df_one_row_per_game['WIN_PCT_BEFORE_AWAY'].isna(), 
        1, 
        df_one_row_per_game['WIN_PCT_BEFORE_HOME'] >= df_one_row_per_game['WIN_PCT_BEFORE_AWAY']
    )
)
df_one_row_per_game.head()



,GAME_ID,GAME_DATE,HOME_TEAM,PTS_home,AWAY_TEAM,PTS_away,IS_NEUTRAL,HOME_WIN,HOME_MARGIN,HOME_TEAM_FULL_SEASON_win_pct,AWAY_TEAM_FULL_SEASON_win_pct,FULL_SEASON_pred,WIN_PCT_BEFORE_HOME,WIN_PCT_BEFORE_AWAY,rolling_history_pred
0,22401187,2025-04-13,BOS,93,CHA,86,False,1,7,0.743902,0.231707,True,0.740741,0.234568,1
1,22401197,2025-04-13,SAS,125,TOR,118,False,1,7,0.414634,0.365854,True,0.407407,0.370370,1
2,22401196,2025-04-13,NOP,100,OKC,115,False,0,-15,0.256098,0.829268,False,0.259259,0.827160,0
3,22401190,2025-04-13,MIA,118,WAS,119,False,0,-1,0.451220,0.219512,True,0.456790,0.209877,1
4,22401189,2025-04-13,CLE,118,IND,126,False,0,-8,0.780488,0.609756,True,0.790123,0.604938,1


In [20]:
(df_one_row_per_game['HOME_WIN'] == df_one_row_per_game['rolling_history_pred']).mean()

np.float64(0.6398373983739838)